# BÀI THỰC HÀNH LAB 05: NỀN TẢNG XÁC SUẤT & TÍNH TOÁN XÁC SUẤT
## Môn học: Xác suất Thống kê (UET.MAT1052) — Trường Đại học Công nghệ, ĐHQGHN
### Nhóm tác giả: Hoàng Thị Điệp, Nghiêm Nguyễn Việt Dũng, Lê Thị Hường
---
### Mục tiêu bài thực hành:
1. Lập trình mô phỏng Monte Carlo vector hóa (NumPy) kiểm chứng Nghịch lý Chevalier de Méré (1654).
2. Thao tác dữ liệu bảng chéo hai chiều (`pd.crosstab`) để tính xác suất biên duyên, đồng thời và có điều kiện.
3. Định lượng các chỉ số y sinh: Độ nhạy (Sensitivity), Độ đặc hiệu (Specificity), Dương tính giả (FPR), và Hậu nghiệm Bayes (PPV).
4. Trực quan hóa dữ liệu chuẩn Grammar of Graphics với thư viện `plotnine`.

### 1. Khởi động môi trường và thiết lập Seed
Nạp các thư viện chuẩn `numpy`, `pandas`, `plotnine` và cố định `seed = 2026` để đảm bảo tính tái lập (Reproducibility).

In [ ]:
import os
import numpy as np
import pandas as pd
from plotnine import (
    ggplot, aes, geom_line, geom_hline, geom_bar,
    labs, theme_minimal, theme, element_text, scale_y_continuous,
    scale_x_continuous, scale_fill_manual
)

SEED = 2026
rng = np.random.default_rng(SEED)
print("Môi trường đã sẵn sàng!")

### 2. Bài 1: Mô phỏng Monte Carlo Nghịch lý Chevalier De Méré (50,000 ván)
- **Trò 1:** Gieo 1 xúc xắc 4 lần độc lập. Thắng nếu có ít nhất một mặt 6. Lý thuyết: $P(B_1) = 1 - (5/6)^4 \approx 51.77\%$.
- **Trò 2:** Gieo 2 xúc xắc 24 lần độc lập. Thắng nếu có ít nhất một cặp (6, 6). Lý thuyết: $P(B_2) = 1 - (35/36)^{24} \approx 49.14\%$.

In [ ]:
n_games = 50_000

# Trò 1: ma trận (50000, 4) chứa các giá trị ngẫu nhiên từ 1 đến 6
game1_rolls = rng.integers(1, 7, size=(n_games, 4))
game1_win = (game1_rolls == 6).any(axis=1).astype(int)

# Trò 2: ma trận (50000, 24, 2)
game2_rolls = rng.integers(1, 7, size=(n_games, 24, 2))
game2_win = ((game2_rolls == 6).all(axis=2)).any(axis=1).astype(int)

p1_sim = game1_win.mean()
p2_sim = game2_win.mean()

p1_theory = 1 - (5 / 6) ** 4
p2_theory = 1 - (35 / 36) ** 24

print(f"Trò 1 (4 lần gieo): Thực nghiệm = {p1_sim:.4f} | Lý thuyết = {p1_theory:.4f} | Sai số = {abs(p1_sim - p1_theory):.4f}")
print(f"Trò 2 (24 lần gieo): Thực nghiệm = {p2_sim:.4f} | Lý thuyết = {p2_theory:.4f} | Sai số = {abs(p2_sim - p2_theory):.4f}")

#### Trực quan hóa sự hội tụ theo Luật Số Lớn (LLN) với `plotnine`

In [ ]:
step = 50
sample_indices = np.arange(step, n_games + 1, step)

cum_win1 = np.cumsum(game1_win)[sample_indices - 1] / sample_indices
cum_win2 = np.cumsum(game2_win)[sample_indices - 1] / sample_indices

df_convergence = pd.DataFrame({
    "game_index": np.tile(sample_indices, 2),
    "cum_rate": np.concatenate([cum_win1, cum_win2]),
    "game_type": ["Trò 1 (4 lần gieo)"] * len(sample_indices) + ["Trò 2 (24 lần gieo)"] * len(sample_indices)
})

p_conv = (
    ggplot(df_convergence, aes(x="game_index", y="cum_rate", color="game_type"))
    + geom_line(size=1)
    + geom_hline(yintercept=0.50, linetype="dashed", color="#ef4444", size=0.9)
    + geom_hline(yintercept=p1_theory, linetype="dotted", color="#0284c7", size=0.8)
    + geom_hline(yintercept=p2_theory, linetype="dotted", color="#ea580c", size=0.8)
    + scale_y_continuous(labels=lambda l: [f"{val * 100:.1f}%" for val in l], limits=[0.44, 0.58])
    + scale_x_continuous(labels=lambda l: [f"{int(val):,}" for val in l])
    + labs(
        title="Kiểm Chứng Nghịch Lý Chevalier De Méré Qua 50,000 Ván Mô Phỏng",
        subtitle="Đường nét đứt đỏ: Ngưỡng hòa vốn 50% | Đường chấm: Xác suất lý thuyết",
        x="Số ván chơi tích lũy (N)",
        y="Tần suất thắng tích lũy",
        color="Loại trò chơi"
    )
    + theme_minimal()
    + theme(
        plot_title=element_text(weight="bold", size=13),
        legend_position="bottom"
    )
)
p_conv

### 3. Bài 2: Phân Tích Bảng Chéo Tần Số & Xác Suất Điều Kiện Trên 1,000 Bệnh Nhân
Nạp tập dữ liệu `du-lieu/medical_screening.csv`.

In [ ]:
df_med = pd.read_csv("du-lieu/medical_screening.csv")
print(f"Số dòng: {len(df_med)}, Số cột: {len(df_med.columns)}")
df_med.head()

In [ ]:
# 1. Bảng tần số số lượng đếm tuyệt đối
ct_counts = pd.crosstab(
    df_med["actual_infection"],
    df_med["test_result"],
    rownames=["Thực tế (Infection)"],
    colnames=["Kết quả Test"],
    margins=True,
    margins_name="Tổng"
)
ct_counts

In [ ]:
# 2. Xác suất có điều kiện theo hàng P(Test | Infection) -> Độ nhạy & Độ đặc hiệu
ct_cond_row = pd.crosstab(
    df_med["actual_infection"],
    df_med["test_result"],
    rownames=["Thực tế"],
    colnames=["Test"],
    normalize="index"
)
print("--- Xác suất có điều kiện theo hàng P(Test | Infection) ---")
ct_cond_row.round(4)

In [ ]:
# 3. Xác suất có điều kiện theo cột P(Infection | Test) -> Giá trị dự báo dương tính PPV
ct_cond_col = pd.crosstab(
    df_med["actual_infection"],
    df_med["test_result"],
    rownames=["Thực tế"],
    colnames=["Test"],
    normalize="columns"
)
print("--- Xác suất có điều kiện theo cột P(Infection | Test) ---")
ct_cond_col.round(4)

In [ ]:
# Trích xuất các chỉ số định lượng y sinh
tp = ((df_med["actual_infection"] == 1) & (df_med["test_result"] == 1)).sum()
fn = ((df_med["actual_infection"] == 1) & (df_med["test_result"] == 0)).sum()
fp = ((df_med["actual_infection"] == 0) & (df_med["test_result"] == 1)).sum()
tn = ((df_med["actual_infection"] == 0) & (df_med["test_result"] == 0)).sum()

prevalence = (df_med["actual_infection"] == 1).mean()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
false_positive_rate = fp / (fp + tn)
ppv_bayes = tp / (tp + fp)

print("=== BẢNG CHỈ SỐ Y SINH ĐỊNH LƯỢNG ===")
print(f"• Tỷ lệ mắc bệnh (Prevalence)     : {prevalence * 100:.2f}%")
print(f"• Độ nhạy (Sensitivity)           : {sensitivity * 100:.2f}%")
print(f"• Độ đặc hiệu (Specificity)       : {specificity * 100:.2f}%")
print(f"• Tỷ lệ dương tính giả (FPR)      : {false_positive_rate * 100:.2f}%")
print(f"• Dương tính thật hậu nghiệm (PPV): {ppv_bayes * 100:.2f}%")

In [ ]:
# Trực quan hóa phân bố kết quả test theo nhóm tuổi bằng plotnine
df_med["Test_Label"] = df_med["test_result"].map({1: "Dương tính (+)", 0: "Âm tính (-)"})
p_age = (
    ggplot(df_med, aes(x="age_group", fill="Test_Label"))
    + geom_bar(position="fill", width=0.6)
    + scale_fill_manual(values=["#0284c7", "#ea580c"])
    + scale_y_continuous(labels=lambda l: [f"{val * 100:.0f}%" for val in l])
    + labs(
        title="Tỷ Lệ Kết Quả Xét Nghiệm Nhanh Theo Từng Nhóm Tuổi",
        subtitle="Biểu đồ cột tỷ lệ chồng (100% Stacked Bar Chart)",
        x="Nhóm tuổi",
        y="Tỷ lệ phân bố",
        fill="Kết quả Test"
    )
    + theme_minimal()
    + theme(legend_position="bottom")
)
p_age

### 4. Bài 3: Khảo Sát Độ Nhạy Hậu Nghiệm Bayes (Sensitivity Analysis)
Khảo sát mối quan hệ phi tuyến giữa tỷ lệ tiên nghiệm $P(D)$ và xác suất hậu nghiệm $P(D \mid +)$ khi độ nhạy = 96% và độ đặc hiệu = 94%.

In [ ]:
def bayes_posterior(prevalence, sensitivity=0.96, specificity=0.94):
    numerator = prevalence * sensitivity
    denominator = numerator + (1 - prevalence) * (1 - specificity)
    return numerator / denominator

prev_grid = np.linspace(0.001, 0.20, 200)
post_grid = bayes_posterior(prev_grid, sensitivity=0.96, specificity=0.94)

df_bayes_curve = pd.DataFrame({
    "prevalence": prev_grid,
    "posterior": post_grid
})

p_bayes = (
    ggplot(df_bayes_curve, aes(x="prevalence", y="posterior"))
    + geom_line(color="#ea580c", size=1.2)
    + geom_hline(yintercept=0.50, linetype="dashed", color="#64748b", size=0.8)
    + scale_x_continuous(labels=lambda l: [f"{val * 100:.1f}%" for val in l])
    + scale_y_continuous(labels=lambda l: [f"{val * 100:.1f}%" for val in l], limits=[0, 1])
    + labs(
        title="Đường Cong Hậu Nghiệm Bayes: P(Bệnh | Test Dương Tính)",
        subtitle="Độ nhạy = 96%, độ đặc hiệu = 94% | Đường nét đứt: Mốc tin cậy 50%",
        x="Tỷ lệ mắc bệnh trong cộng đồng P(D) (Prior)",
        y="Xác suất thực sự mắc bệnh P(D | +) (Posterior)"
    )
    + theme_minimal()
)
p_bayes

### 5. Kết Luận & Bài Học Thực Tiễn
1. **Luật Số Lớn:** Khi $N$ tăng lên 50,000 ván, tần suất thực nghiệm hội tụ rất sát với xác suất lý thuyết (sai số $< 0.5\%$).
2. **Nghịch lý Dương tính giả:** Khi một căn bệnh là hiếm trong cộng đồng ($P(D) < 5\%$), ngay cả một xét nghiệm có độ nhạy và đặc hiệu rất cao (trên $94\%$) thì một người có kết quả dương tính vẫn có xác suất cao là không mắc bệnh. Do đó trong y tế, cần phải xét nghiệm khẳng định (Confirmatory test) lần hai trước khi đưa ra chẩn đoán cuối cùng!